In [ ]:
import numpy as np
import casadi as cs
import matplotlib.pyplot as plt
import time

np.random.seed(42)

# ============================================================
# 1. Simulation & System Parameters
# ============================================================
h_dt = 0.01  
v_target = 10.0
k_vel = 0.1
R_meas_std = 1.5
TRUE_BIAS = 1.5

# Lateral dynamics (Ornstein-Uhlenbeck process)
k_damp = 1.0    
k_restore = 0.5 

# Noise covariance matrices
Q = np.diag([0.001, 0.01, 0.001, 0.01, 1e-6])
R_mat = (R_meas_std**2) * np.eye(4)

# Initial state and covariance
x0 = np.array([[0.0], [10.0], [0.0], [0.0], [0.0]])
P0 = np.diag([1.0, 0.5, 0.5, 0.1, 2.0])

# Rectangular beacon layout (Bounding box)
beacons = np.array([
    [-60.0, 60.0],   # B1: Top-Left
    [-60.0, -60.0],  # B2: Bottom-Left
    [130.0, 60.0],   # B3: Top-Right (Target for bias injection)
    [130.0, -60.0]   # B4: Bottom-Right
])
ny = len(beacons)

# ============================================================
# 2. Complex Track Model & CasADi Automatic Differentiation
# ============================================================
R1, R2, d_centers = 48.0, 22.0, 100.0
sin_alpha = (R1 - R2) / d_centers
alpha = np.arcsin(sin_alpha)

L_str = d_centers * np.cos(alpha)
L_curveB = R2 * (np.pi - 2 * alpha) 
L_curveA = R1 * (np.pi + 2 * alpha) 
L_tot = 2 * L_str + L_curveB + L_curveA

P_A_top_x = R1 * np.sin(alpha)
P_A_top_y = R1 * np.cos(alpha)
P_B_bot_x = d_centers + R2 * np.sin(alpha)
P_B_bot_y = -R2 * np.cos(alpha)

# CasADi symbolic variables
s_sym, vs_sym = cs.MX.sym('s'), cs.MX.sym('vs')
d_sym, vd_sym = cs.MX.sym('d'), cs.MX.sym('vd')
b_sym = cs.MX.sym('b')
state_sym = cs.vertcat(s_sym, vs_sym, d_sym, vd_sym, b_sym)

s_mod = cs.fmod(s_sym, L_tot)

# Segment definitions
x1 = P_A_top_x + s_mod * cs.cos(-alpha)
y1 = P_A_top_y + s_mod * cs.sin(-alpha)
psi1 = -alpha

s2 = s_mod - L_str
theta2 = (np.pi/2 - alpha) - s2 / R2
x2 = d_centers + R2 * cs.cos(theta2)
y2 = R2 * cs.sin(theta2)
psi2 = theta2 - np.pi/2

s3 = s_mod - (L_str + L_curveB)
x3 = P_B_bot_x + s3 * cs.cos(-np.pi + alpha)
y3 = P_B_bot_y + s3 * cs.sin(-np.pi + alpha)
psi3 = -np.pi + alpha

s4 = s_mod - (2 * L_str + L_curveB)
theta4 = (-np.pi/2 + alpha) - s4 / R1
x4 = R1 * cs.cos(theta4)
y4 = R1 * cs.sin(theta4)
psi4 = theta4 - np.pi/2

# Piecewise track mapping
x_c = cs.if_else(s_mod < L_str, x1, 
        cs.if_else(s_mod < L_str + L_curveB, x2, 
          cs.if_else(s_mod < 2 * L_str + L_curveB, x3, x4)))
y_c = cs.if_else(s_mod < L_str, y1, 
        cs.if_else(s_mod < L_str + L_curveB, y2, 
          cs.if_else(s_mod < 2 * L_str + L_curveB, y3, y4)))
psi_c = cs.if_else(s_mod < L_str, psi1, 
          cs.if_else(s_mod < L_str + L_curveB, psi2, 
            cs.if_else(s_mod < 2 * L_str + L_curveB, psi3, psi4)))

# Global coordinates with lateral offset
X_global = x_c - d_sym * cs.sin(psi_c)
Y_global = y_c + d_sym * cs.cos(psi_c)

# Measurement model with augmented bias
h_list = []
for i in range(ny):
    dist = cs.sqrt((X_global - beacons[i, 0])**2 + (Y_global - beacons[i, 1])**2)
    h_list.append(dist + b_sym if i == 2 else dist)
h_sym = cs.vertcat(*h_list)

# CasADi function generation
calc_global_pos = cs.Function('calc_global_pos', [s_sym, d_sym], [X_global, Y_global])
h_func = cs.Function('h_func', [state_sym], [h_sym])
C_func = cs.Function('C_func', [state_sym], [cs.jacobian(h_sym, state_sym)])

# ============================================================
# 3. Discrete-Time System Dynamics
# ============================================================
def f_discrete(x, dt):
    s, v_s, d, v_d, b = x.flatten()
    return np.array([
        [s + v_s * dt],
        [v_s + k_vel * (v_target - v_s) * dt],
        [d + v_d * dt],
        [v_d - (k_damp * v_d + k_restore * d) * dt],
        [b]
    ])

def get_A_matrix(x, dt):
    A = np.eye(5)
    A[0, 1] = dt
    A[1, 1] = 1 - k_vel * dt
    A[2, 3] = dt
    A[3, 2] = -k_restore * dt
    A[3, 3] = 1 - k_damp * dt
    return A

# ============================================================
# 4. Extended Kalman Filter (EKF) Main Loop
# ============================================================
n_steps = 4000
x_true = x0.copy()
x_est = x0.copy()
P_est = P0.copy()

true_history, est_history, P_history = [], [], []

start_exec = time.time()

for k in range(n_steps):
    # --- True Plant Simulation ---
    w_true = np.random.multivariate_normal(np.zeros(5), Q).reshape(-1, 1)
    w_true[4, 0] = 0
    x_true = f_discrete(x_true, h_dt) + w_true
    x_true[4, 0] = TRUE_BIAS
    
    # Hard boundary enforcement for the true plant
    if x_true[2, 0] > 2.0: 
        x_true[2, 0] = 2.0
        x_true[3, 0] *= -0.5
    elif x_true[2, 0] < -2.0: 
        x_true[2, 0] = -2.0
        x_true[3, 0] *= -0.5

    # --- Measurement Generation ---
    z_real = np.array(h_func(x_true)).reshape(-1, 1) + np.random.normal(0, R_meas_std, (ny, 1))

    # --- EKF Time Update (Prediction) ---
    x_pred = f_discrete(x_est, h_dt)
    A_k = get_A_matrix(x_est, h_dt)
    P_pred = A_k @ P_est @ A_k.T + Q
    P_pred = (P_pred + P_pred.T) / 2.0

    # --- EKF Measurement Update (Correction) ---
    C_k = np.array(C_func(x_pred)).astype(float)
    z_hat = np.array(h_func(x_pred)).reshape(-1, 1)
    
    innovation = z_real - z_hat
    S = C_k @ P_pred @ C_k.T + R_mat
    K = P_pred @ C_k.T @ np.linalg.inv(S)
    
    x_est = x_pred + K @ innovation
    P_est = (np.eye(5) - K @ C_k) @ P_pred
    P_est = (P_est + P_est.T) / 2.0

    # Data logging
    true_history.append(x_true.flatten())
    est_history.append(x_est.flatten())
    P_history.append(P_est.copy())

exec_time = (time.time() - start_exec) / n_steps

# ============================================================
# 5. Performance Metrics Evaluation
# ============================================================
true_history = np.array(true_history)
est_history = np.array(est_history)
P_history = np.array(P_history)
t_axis = np.arange(n_steps) * h_dt

# Calculate RMSE excluding the initial burn-in period (100 steps)
rmse_s = np.sqrt(np.mean((true_history[100:, 0] - est_history[100:, 0])**2))
rmse_d = np.sqrt(np.mean((true_history[100:, 2] - est_history[100:, 2])**2))
bias_error = abs(TRUE_BIAS - est_history[-1, 4])

print("="*50)
print("STEP 4: FINAL ESTIMATION METRICS")
print("="*50)
print(f"Avg Execution Time : {exec_time:.6f} s/step")
print(f"Longitudinal RMSE  : {rmse_s:.4f} m")
print(f"Lateral RMSE       : {rmse_d:.4f} m")
print(f"Final Bias Est.    : {est_history[-1, 4]:.4f} m (Error: {bias_error:.4f} m)")
print("="*50)

# ============================================================
# 6. Result Visualization
# ============================================================

# [Figure 1] Lateral Tracking Performance
plt.figure(figsize=(10, 5))
sigma_d = np.sqrt(P_history[:, 2, 2])
plt.plot(t_axis, true_history[:, 2], 'b-', linewidth=1.2, alpha=0.8, label='True Lateral Offset')
plt.plot(t_axis, est_history[:, 2], 'r--', linewidth=1.2, alpha=0.9, label='Estimated Offset (EKF)')
plt.fill_between(t_axis, est_history[:, 2] - 3*sigma_d, est_history[:, 2] + 3*sigma_d, color='red', alpha=0.2, label=r'$\pm 3\sigma$ Bound')

plt.axhline(2.0, color='k', linestyle='-', linewidth=1, alpha=0.7, label='Lane Boundary (+2m)')
plt.axhline(-2.0, color='k', linestyle='-', linewidth=1, alpha=0.7, label='Lane Boundary (-2m)')

plt.title(f'Lateral Tracking Performance (RMSE: {rmse_d:.4f} m)')
plt.xlabel('Time [s]')
plt.ylabel('Lateral Offset [m]')
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# [Figure 2] Sensor Bias Convergence
plt.figure(figsize=(10, 5))
sigma_b = np.sqrt(P_history[:, 4, 4])
plt.plot(t_axis, est_history[:, 4], 'g-', linewidth=2, label='Estimated Bias')
plt.fill_between(t_axis, est_history[:, 4] - 3*sigma_b, est_history[:, 4] + 3*sigma_b, color='green', alpha=0.2, label=r'$\pm 3\sigma$ Bound')
plt.axhline(TRUE_BIAS, color='r', linestyle='--', label=f'True Injected Bias ({TRUE_BIAS}m)')

# Restrict Y-axis to prevent initial massive variance from flattening the plot
plt.ylim([0.0, 3.0]) 

plt.title('Sensor Bias Convergence (Augmented State 5)')
plt.xlabel('Time [s]')
plt.ylabel('Bias [m]')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# [Figure 3] 2D Trajectory with Track Boundaries
plt.figure(figsize=(10, 6))

s_vals = np.linspace(0, L_tot, 1000)
X_c, Y_c, X_in, Y_in, X_out, Y_out = [], [], [], [], [], []
for s_val in s_vals:
    xc, yc = calc_global_pos(float(s_val), 0.0) 
    xin, yin = calc_global_pos(float(s_val), -2.0) 
    xout, yout = calc_global_pos(float(s_val), 2.0) 
    
    X_c.append(float(xc)); Y_c.append(float(yc))
    X_in.append(float(xin)); Y_in.append(float(yin))
    X_out.append(float(xout)); Y_out.append(float(yout))

plt.plot(X_c, Y_c, 'k--', linewidth=0.5, alpha=0.4, label='Track Centerline')
plt.plot(X_in, Y_in, 'k-', linewidth=1, alpha=0.7, label='Track Boundaries (±2m)')
plt.plot(X_out, Y_out, 'k-', linewidth=1, alpha=0.7)

X_true_arr, Y_true_arr, X_est_arr, Y_est_arr = [], [], [], []
for i in range(0, n_steps, 5):
    xt, yt = calc_global_pos(float(true_history[i, 0]), float(true_history[i, 2]))
    xe, ye = calc_global_pos(float(est_history[i, 0]), float(est_history[i, 2]))
    
    X_true_arr.append(float(xt)); Y_true_arr.append(float(yt))
    X_est_arr.append(float(xe)); Y_est_arr.append(float(ye))

plt.plot(X_true_arr, Y_true_arr, 'b-', linewidth=0.8, alpha=0.9, label='True Trajectory')
plt.plot(X_est_arr, Y_est_arr, 'r--', linewidth=0.8, alpha=0.9, label='Estimated Trajectory')

plt.scatter(beacons[:, 0], beacons[:, 1], c='red', marker='^', s=150, edgecolors='black', label='Beacons')
for i, (bx, by) in enumerate(beacons):
    label = f'B{i+1} (Bias: 1.5m)' if i == 2 else f'B{i+1}'
    plt.text(bx + 3, by + 3, label, fontsize=10, fontweight='bold', 
             bbox=dict(facecolor='white', alpha=0.6, edgecolor='none'))

plt.title('2D Trajectory with Full Rectangular Beacon Coverage')
plt.xlabel('Global X [m]')
plt.ylabel('Global Y [m]')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.legend(loc='center', bbox_to_anchor=(0.5, 0.5), fontsize=9)
plt.show()